# MatchMiner-AI example workflow

This notebook runs the matching workflow on the sample data in `data/`: ClinicalTrials.gov listings
(`trials_for_summarize.csv`) and the notes of twelve synthetic patients (`note_set_1.csv`, with later notes in
`note_set_2.csv`). It goes from the two tables to scored patient-cohort pairs:

1. **Trials to cohorts** (`process_trials`): screen each trial for cancer treatment, extract its cohorts and their
   criteria, merge equivalent cohorts and tag every criteria line.
2. **Notes to patient summaries** (`summarize_patients`): serial summarization into a cancer history and the other
   relevant history, then an update with newer notes.
3. **Embed and retrieve** (`embed_for_matching`, `generate_candidate_matches`): TrialSpace vectors and the nearest
   cohorts for each patient.
4. **Score the pairs** with the decision model: match quality, boilerplate criteria, and every criteria line.
5. **Optional LLM checks** that return a reason with each answer.
6. **A review table**, one row per patient-cohort pair.

**You need:**
- a GPU for TrialSpace and the decision model;
- a TrialSpace checkpoint;
- an LLM for steps 1 and 2: in-process with vLLM, or an OpenAI-compatible server.

Run the notebook from the `examples/` directory.

Scores rank and screen candidates for human review. They are not eligibility decisions.

In [ ]:
import json
import os
from pathlib import Path

import pandas as pd

# Optional environment setup, before any model loads:
# os.environ["CUDA_VISIBLE_DEVICES"] = "0"   # the GPUs this notebook may use
# os.environ["HF_TOKEN"] = "hf_..."          # higher Hugging Face Hub rate limits

## Configuration

Everything comes from the default preset. Set two things before running:

- **`embedding.model_path`:** your TrialSpace checkpoint. The preset names a local checkpoint that is not
  published, and embedding fails rather than fall back to another model.
- **The LLM:**
    - By default `process_trials` and `summarize_patients` run the preset's model
      (`nvidia/NVIDIA-Nemotron-3.5-Super-VL-120B-A12B-NVFP4`) in-process with vLLM.
    - To use a running OpenAI-compatible server instead, set `USE_SERVER = True` and its URL. The server must serve
      the preset's model, or you change the sections' `model_name`.

Only send clinical text to an endpoint approved for it. The sample patients are synthetic.

In [ ]:
from matchminer_ai import load_preset

config = load_preset("default")
config.embedding["model_path"] = "/path/to/trialspace"   # your TrialSpace checkpoint

USE_SERVER = False
if USE_SERVER:
    config.remote["enabled"] = True
    config.remote["server_urls"] = ["http://localhost:8000/v1"]
    os.environ.setdefault("OPENAI_API_KEY", "not-needed")

output_dir = Path("output")
output_dir.mkdir(exist_ok=True)

Optionally, start a local vLLM server for the preset's model and use it as the server above. The trial and patient
steps use the same model, so one server serves both. Stop it at the end with `server.terminate()`.

In [ ]:
# from matchminer_ai.llm.vllm_server import start_vllm_server
#
# server = start_vllm_server(config=config, task="trial")   # waits until /v1/models answers
# config.remote["enabled"] = True
# config.remote["server_urls"] = ["http://localhost:8000/v1"]
# os.environ.setdefault("OPENAI_API_KEY", "not-needed")

## 1. Trials to cohorts

`process_trials` runs every trial step on ClinicalTrials.gov listings:
1. a cancer-treatment screen;
2. the trial's cohorts, with each cohort's complete eligibility criteria;
3. merging cohorts whose criteria are equivalent;
4. tagging each criteria line as inclusion or exclusion and as key or boilerplate.

The example uses the first ten trials. With `output_dir`, the merge and line-tagging results are saved per
trial and reused while the inputs and prompts are unchanged; the listings' cohorts are extracted again on each run.

In [ ]:
trials = pd.read_csv("data/trials_for_summarize.csv").head(10)

from matchminer_ai.trials import process_trials

cohorts, trial_metadata = process_trials(
    trials, config=config, output_dir=output_dir / "trial_runs", return_metadata=True
)
print(f"{trials['trial_id'].nunique()} trials -> {len(cohorts)} cohorts")

One row per cohort:

| Column | Content |
| --- | --- |
| `cohort_trial_id` | Identifier, `<trial_id>-cohort-<n>` |
| `cohort_eligibility_criteria` | The complete criteria |
| `cohort_key_criteria` | What defines the population; read by TrialSpace and the match questions |
| `cohort_boilerplate_criteria` | The general criteria most trials share; read by the boilerplate question |
| `cohort_criteria_items` | Every line with its tags; read by the line-by-line check |

In [ ]:
cohorts[["cohort_trial_id", "trial_id", "cohort_name", "cohort_key_criteria", "cohort_boilerplate_criteria"]].head()

Trials that are not cancer treatment trials have no cohorts; the screen's decisions are in the metadata.

In [ ]:
trial_metadata["extraction"]["trial_rejections"]

In [ ]:
cohorts.to_parquet(output_dir / "cohorts.parquet", index=False)

## 2. Notes to patient summaries

`summarize_patients` summarizes each patient's notes serially.
- **Preparation:** the notes are sorted by date, repeated sentences are removed, and the record is split into
  overlapping chunks.
- **Updates:** each chunk updates the running summary.
- **Result:** one row per patient. `cancer_history_summary` is the key section, and
  `general_exclusion_criteria_evidence` is the evidence for boilerplate criteria.
- **Resuming:** `checkpoint_dir` saves each chunk as it finishes, so an interrupted run resumes.
- **QC report:** counts any patient whose summary kept failing the quality checks; that patient is left out.

In [ ]:
from matchminer_ai.patients import summarize_patients

notes = pd.read_csv("data/note_set_1.csv", dtype={"patient_id": str})

patients, patient_metadata, patient_qc = summarize_patients(
    notes,
    config=config,
    checkpoint_dir=output_dir / "checkpoints" / "patients",
    return_metadata=True,
    return_qc=True,
)
patients[["patient_id", "last_note_date", "cancer_history_summary", "general_exclusion_criteria_evidence"]].head()

In [ ]:
patient_qc

### Updating summaries with newer notes

Pass the summaries back as `existing_summaries` with the new notes. With `last_note_date`, only notes after each
summary's last note are added.

In [ ]:
new_notes = pd.read_csv("data/note_set_2.csv", dtype={"patient_id": str})

patients, patient_metadata, patient_qc = summarize_patients(
    new_notes,
    config=config,
    existing_summaries=patients[
        ["patient_id", "last_note_date", "cancer_history_summary", "general_exclusion_criteria_evidence"]
    ],
    checkpoint_dir=output_dir / "checkpoints" / "patients_update",
    return_metadata=True,
    return_qc=True,
)
patients.to_parquet(output_dir / "patient_summaries.parquet", index=False)
patients[["patient_id", "last_note_date", "cancer_history_summary"]].head()

## 3. Embed and retrieve

TrialSpace encodes patients' cancer histories and cohorts' key criteria into one vector space. Retrieval ranks by
cosine similarity in either direction. Never compare vectors made by different TrialSpace models.

To restart from here, uncomment the first two lines to read the saved tables.

In [ ]:
# cohorts = pd.read_parquet(output_dir / "cohorts.parquet")
# patients = pd.read_parquet(output_dir / "patient_summaries.parquet")

from matchminer_ai.embedding import embed_for_matching
from matchminer_ai.matching import generate_candidate_matches

patient_vectors = embed_for_matching(patients, entity_type="patient", config=config)
cohort_vectors = embed_for_matching(cohorts, entity_type="trial", config=config)

# Patient-centric: the 20 nearest cohorts for each patient.
candidates = generate_candidate_matches(patient_vectors, cohort_vectors, k=20)
candidates.head()

Trial-centric matching swaps the two tables: the nearest patients for each cohort.

In [ ]:
generate_candidate_matches(cohort_vectors, patient_vectors, k=5).head()

## 4. Score the pairs with the decision model

The decision model answers yes/no questions about each pair in one forward pass. Each check reads the pair's texts,
so first join the summaries and cohorts onto the candidates.

- **`score_match_quality`:** five match questions (reasonableness, cancer type, disease burden, treatment history,
  biomarkers).
    - `match_quality_score` ranks the pairs.
    - `match_quality_pass` is true when the model calls the pair potentially eligible.
    - `filter_low_quality=False` keeps every pair; by default only passing pairs are returned.
- **`exclusion_criteria_check`:** whether the patient clearly fails a boilerplate criterion.
- **`check_eligibility_criteria`:** whether the patient clearly fails each criteria line; one row per line.

In [ ]:
from matchminer_ai.matching import check_eligibility_criteria, exclusion_criteria_check, score_match_quality

keys = ["patient_id", "cohort_trial_id"]
pairs = candidates.merge(patients, on="patient_id").merge(cohorts, on="cohort_trial_id")

quality = score_match_quality(pairs, config=config, filter_low_quality=False)
pairs = pairs.merge(quality, on=keys)
pairs["match_quality_pass"].value_counts()

In [ ]:
boilerplate = exclusion_criteria_check(pairs, config=config)
boilerplate["exclusion_criteria_pass"].value_counts()

In [ ]:
passing = pairs[pairs["match_quality_pass"]]
failing_lines = check_eligibility_criteria(passing, config=config, filter_failing=True)
failing_lines[keys + ["criterion_number", "criterion_type", "criterion_text", "criterion_pass_probability"]].head()

## 5. Optional: LLM checks

Each decision-model check has an LLM version, which returns a reason with each answer.
`score_match_quality_with_llm` and `check_eligibility_criteria_with_llm` use the training teacher's prompts. They
send one request per pair, or per criteria line, so the example asks them about a few passing pairs only.

In [ ]:
from matchminer_ai.matching import (
    check_eligibility_criteria_with_llm,
    exclusion_criteria_check_with_llm,
    score_match_quality_with_llm,
)

sample = passing.sort_values("match_quality_score", ascending=False).head(5)
llm_quality = score_match_quality_with_llm(sample, config=config)
llm_boilerplate = exclusion_criteria_check_with_llm(sample, config=config)
llm_lines = check_eligibility_criteria_with_llm(sample.head(1), config=config)
llm_quality.head()

## 6. Review table

One row per patient-cohort pair, with:
- the retrieval rank and the decision model's answers;
- the boilerplate check;
- the number of criteria lines the patient likely fails.

Within each patient, rows are ordered by match-quality score. Nothing is filtered out; every flag is shown for a
reviewer.

In [ ]:
lines_failed = failing_lines.groupby(keys).size().rename("criteria_lines_failed").reset_index()

review = (
    pairs[keys + ["trial_id", "cohort_name", "rank", "similarity_score", "match_quality_score", "match_quality_pass"]]
    .merge(boilerplate[keys + ["exclusion_criteria_pass"]], on=keys, how="left")
    .merge(lines_failed, on=keys, how="left")
    .fillna({"criteria_lines_failed": 0})
    .sort_values(["patient_id", "match_quality_score"], ascending=[True, False])
)
review.to_csv(output_dir / "final_matches.csv", index=False)
review.head(10)